# Week 1｜NumPy 基础与五道练习

**目标：**理解数组、形状、索引、广播和向量化。请按顺序运行；每个 `TODO` 先自己完成，再运行其后的验证格。

## 0. 导入库
`np` 是 NumPy 的常用简称。固定随机种子可以保证每次运行得到相同的随机数，方便复现。

In [1]:
import numpy as np
np.random.seed(42)

## 1. 数组、形状与索引
二维数组可以看成表格：先写行号、再写列号；Python 从 0 开始计数。

In [2]:
x = np.arange(24).reshape(4, 6)
print(x)
print('形状:', x.shape)
print('第 0 行:', x[0])
print('第 2 列:', x[:, 2])
print('行 1-2、列 2-4:', x[1:3, 2:5])

[[ 0  1  2  3  4  5]
 [ 6  7  8  9 10 11]
 [12 13 14 15 16 17]
 [18 19 20 21 22 23]]
形状: (4, 6)
第 0 行: [0 1 2 3 4 5]
第 2 列: [ 2  8 14 20]
行 1-2、列 2-4: [[ 8  9 10]
 [14 15 16]]


## 2. 广播：不写循环地对每一列运算
`keepdims=True` 会让均值保持为 `(1, 5)`，因此它能自动广播到 `(100, 5)` 的每一行。

In [6]:
demo = np.array([[1., 2., 3.], [4., 5., 6.]])
print(demo)
列均值 = demo.mean(axis=0, keepdims=True)
print('列均值:', 列均值)
print('每列减均值后:\n', demo - 列均值)

[[1. 2. 3.]
 [4. 5. 6.]]
列均值: [[2.5 3.5 4.5]]
每列减均值后:
 [[-1.5 -1.5 -1.5]
 [ 1.5  1.5  1.5]]


## 练习 1：z-score 标准化
目标：让每一列的均值约为 0、标准差约为 1。提示：`axis=0` 表示沿行方向聚合，即分别处理每一列。

In [14]:
data = np.random.randn(100, 5)

# TODO：补全下面两行（注意 keepdims=True）保留维度：便于广播和向量化。。
mean = mean = np.mean(data, axis=0, keepdims=True)
std = np.std(data, axis=0, keepdims=True)
data_normalized = (data - mean) / std
data_normalized[:3]

array([[ 0.55371488,  0.88090616, -0.22165102, -0.72695875, -1.23336879],
       [-0.75701086,  2.76486372,  1.39479201, -1.5670484 , -1.46449305],
       [ 0.47039043,  0.48058749,  0.68113543, -0.3207152 , -0.2986779 ]])

In [15]:
# 验证：理想输出应都非常接近 0 和 1。之所以不是0，是因为浮点数精度误差
print('每列均值:', data_normalized.mean(axis=0))
print('每列标准差:', data_normalized.std(axis=0))

每列均值: [-3.87190280e-17 -5.96744876e-17 -2.22044605e-17  2.38697950e-17
  8.99280650e-17]
每列标准差: [1. 1. 1. 1. 1.]


## 练习 2：余弦相似度矩阵
先把每个向量除以自己的 L2 范数，得到单位向量；两个单位向量的点积就是余弦相似度。`@` 表示矩阵乘法。

In [16]:
A = np.random.randn(10, 128)

# 每一行的 L2 范数，keepdims=True 保持形状为 (10, 1)
norms = np.linalg.norm(A, axis=1, keepdims=True)

# 每一行归一化为单位向量
A_unit = A / norms

# 两两余弦相似度矩阵，形状为 (10, 10)
similarity = A_unit @ A_unit.T

similarity

array([[ 1.        , -0.05841767, -0.05014619, -0.07946933,  0.11325286,
         0.1822815 , -0.01594217, -0.01253938, -0.08228561,  0.12685333],
       [-0.05841767,  1.        ,  0.02909898,  0.02057529, -0.02612731,
         0.03437359,  0.08672901, -0.04826535, -0.02580279,  0.06716201],
       [-0.05014619,  0.02909898,  1.        , -0.0072135 , -0.0708071 ,
        -0.02302861,  0.15173772,  0.02522846,  0.00379372,  0.06372885],
       [-0.07946933,  0.02057529, -0.0072135 ,  1.        , -0.0381191 ,
         0.09293612, -0.01509954, -0.10917847, -0.01702997,  0.01390251],
       [ 0.11325286, -0.02612731, -0.0708071 , -0.0381191 ,  1.        ,
         0.09968447,  0.14978255,  0.13194582,  0.09471423, -0.20206036],
       [ 0.1822815 ,  0.03437359, -0.02302861,  0.09293612,  0.09968447,
         1.        ,  0.05171141,  0.09307752, -0.05156934, -0.03152881],
       [-0.01594217,  0.08672901,  0.15173772, -0.01509954,  0.14978255,
         0.05171141,  1.        , -0.05597647

In [17]:
# 验证：矩阵应为 (10, 10)，主对角线应约为 1
print('形状:', similarity.shape)
print('主对角线:', np.diag(similarity))

形状: (10, 10)
主对角线: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


## 练习 3：数值稳定的 softmax
直接计算 `exp(1000)` 会溢出。先减去每行最大值，softmax 的结果不会改变，但计算会稳定。

In [18]:
def softmax(x):
    # 转成浮点数组，避免整数计算问题
    x = np.asarray(x, dtype=float)

    # 每一行减去该行最大值，防止 exp 溢出
    x_shifted = x - np.max(x, axis=-1, keepdims=True)

    # 计算指数
    exp_x = np.exp(x_shifted)

    # 每一行归一化，使概率之和为 1
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)


scores = np.array([
    [1., 2., 3.],
    [1000., 1001., 1002.]
])

probabilities = softmax(scores)
probabilities

array([[0.09003057, 0.24472847, 0.66524096],
       [0.09003057, 0.24472847, 0.66524096]])

In [19]:
# 验证：每一行的和应为 1，且不应出现 nan 或 inf
print(probabilities.sum(axis=-1))
assert np.allclose(probabilities.sum(axis=-1), 1.0)

[1. 1.]


## 练习 4：one-hot 编码
类别标签 `2`（共 4 类）应变成 `[0, 0, 1, 0]`。提示：先创建全零矩阵，再用 `result[np.arange(N), labels] = 1` 填入 1。

In [20]:
def one_hot(labels, num_classes):
    # 生成单位矩阵，再用 labels 选择对应的行
    return np.eye(num_classes, dtype=np.float32)[labels]


labels = np.array([0, 2, 1, 3])
encoded = one_hot(labels, num_classes=4)
encoded

array([[1., 0., 0., 0.],
       [0., 0., 1., 0.],
       [0., 1., 0., 0.],
       [0., 0., 0., 1.]], dtype=float32)

In [21]:
# 验证：每行只有一个 1
assert encoded.shape == (4, 4)
assert encoded.dtype == np.float32
assert np.all(encoded.sum(axis=1) == 1)
print('练习 4 通过')

练习 4 通过


## 练习 5：向量化 k-NN 距离
要得到 `(Q, N)` 的距离矩阵，把 `query` 变成 `(Q, 1, D)`，把 `database` 变成 `(1, N, D)`；相减时 NumPy 会自动广播。

In [23]:
def knn_distances(query, database):
    # query:    (M, D)
    # database: (N, D)

    diff = query[:, None, :] - database[None, :, :]  # (M, N, D)
    return np.linalg.norm(diff, axis=-1)              # (M, N)


query = np.array([[0., 0.], [3., 4.]])
database = np.array([[0., 0.], [3., 4.], [6., 8.]])

distances = knn_distances(query, database)
distances

array([[ 0.,  5., 10.],
       [ 5.,  0.,  5.]])

In [24]:
# 验证：第一行应为 [0, 5, 10]，第二行应为 [5, 0, 5]
expected = np.array([[0., 5., 10.], [5., 0., 5.]])
assert np.allclose(distances, expected)
print('五道练习全部完成！')

五道练习全部完成！


## 本 Notebook 小结
请用自己的话补充：
- 我理解的广播是：当两个数组形状不完全相同时，NumPy 会按照一定规则自动扩展较小的数组，使它们能够进行逐元素运算。广播并不一定真的复制数据，而是让不同形状的数组在逻辑上对齐。例如 (10, 128) 的矩阵可以除以 (10, 1) 的每行范数，从而分别归一化每一行。
- 我理解的向量化是：把原本需要使用 Python 循环逐个处理的计算，改写成 NumPy 的数组运算、广播或矩阵乘法，让底层一次处理大量数据。例如计算 k-NN 距离时，可以通过增加维度和广播，一次得到所有查询向量与数据库向量之间的距离矩阵，而不需要写双重 for 循环。
- 我仍然不理解的是：面对一个具体问题时，怎样快速判断应该增加哪个维度、在哪个 axis 上计算，以及广播后数组的形状会如何变化。我还需要通过更多例子练习形状推导，并理解向量化虽然通常更快，但有时也会产生较大的中间数组、占用更多内存。